#  Langfuse Evaluation을 사용한 RAG 답변 평가

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import uuid

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Langsmith tracing 여부를 확인 (true: langsmith 추적 활성화, false: langsmith 추적 비활성화)
print("langsmith 추적 여부: ", os.getenv('LANGSMITH_TRACING'))

`(3) Test Data`

In [ ]:
# Test 데이터셋에 대한 QA 생성 결과를 리뷰한 후 다시 로드
df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

---

## **검색 도구 정의** 

### 1) **벡터스토어** 로드

- **Chroma DB** 설정에서 모델, 컬렉션명, 저장 경로 지정

In [ ]:
# 벡터 저장소 로드 
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

In [ ]:
# 벡터저장소 검색기 생성
chroma_k = chroma_db.as_retriever(
    search_kwargs={'k': 4},
)

# 벡터저장소 검색기를 사용하여 검색
query = "테슬라의 회장은 누구인가요?"

retrieved_docs = chroma_k.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("-"*200)
    print()

### 2) **BM25 검색기** 준비

- **BM25 검색기** 구현으로 문서 유사도 기반 검색 가능

- **한국어 텍스트 처리**를 위한 **Kiwi 토크나이저** 설정

- 참고: https://github.com/bab2min/kiwipiepy

In [ ]:
# korean_docs 파일을 로드 (jsonlines 파일)
def load_jsonlines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs = [json.loads(line) for line in f]
    return docs

korean_docs = load_jsonlines('data/korean_docs_final.jsonl')
print(f"로드된 문서: {len(korean_docs)}개")
pprint(korean_docs[0])

In [ ]:
from langchain.schema import Document  # Document 클래스 임포트

# 문자열 리스트를 Document 객체로 변환
if isinstance(korean_docs[0], str):  # 첫 번째 항목이 문자열인지 확인
    documents = [
        Document(
            page_content=json.loads(data)['page_content'],  # 문자열을 파이썬 객체로 변환
            metadata=json.loads(data)['metadata']
        ) 
        for i, data in enumerate(korean_docs)
    ]
else:
    documents = korean_docs

print(f"변환된 문서: {len(documents)}개")
pprint(documents[0])

In [ ]:
# BM25 검색기를 사용하기 위한 준비
from krag.tokenizers import KiwiTokenizer
from krag.retrievers import KiWiBM25RetrieverWithScore

kiwi_tokenizer = KiwiTokenizer(
    model_type='knlm',    # Kiwi 언어 모델 타입
    typos='basic'         # 기본 오타교정
    )

bm25_db = KiWiBM25RetrieverWithScore(
        documents=documents, 
        kiwi_tokenizer=kiwi_tokenizer, 
        k=4, 
    )

In [ ]:
# BM25 검색기를 사용하여 문서 검색
query = "테슬라의 회장은 누구인가요?"
retrieved_docs = bm25_db.invoke(query)

# 검색 결과 출력 
for doc in retrieved_docs:
    print(f"BM25 점수: {doc.metadata["bm25_score"]:.2f}")    
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)

### 3) **Emsemble Hybrid Search** 준비

- **BM25**, **벡터 검색** 결과를 **rank-fusion** 알고리즘으로 통합 (**EnsembleRetriever**)

- 각 검색기의 **순위 점수**를 고려한 최종 순위 결정

- **중복 문서** 제거와 **재순위화** 자동 수행

- 두 검색 방식의 **장점을 결합**해 검색 품질 향상

In [ ]:
from langchain.retrievers import EnsembleRetriever

# 검색기 초기화 
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_db, chroma_k],
    weights=[0.5, 0.5],
)

In [ ]:
query = "테슬라의 회장은 누구인가요?"
retrieved_docs = hybrid_retriever.invoke(query)

# 검색 결과 출력 
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)

## **RAG Chain** 정의

- OpenAI gpt-4.1-mini 모델 활용

In [ ]:
# 각 쿼리에 대한 검색 결과를 한꺼번에 Context로 전달해서 답변을 생성
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def create_rag_chain(retriever, llm):

    template = """Answer the following question based on this context. If the context is not relevant to the question, just answer with '답변에 필요한 근거를 찾지 못했습니다.'

    [Context]
    {context}

    [Question]
    {question}

    [Answer]
    """

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join([f"{doc.page_content}" for doc in docs])

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()} 
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [ ]:
# RAG 체인 생성 및 테스트
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.5)

openai_rag_chain = create_rag_chain(hybrid_retriever, llm)

question = "테슬라의 회장은 누구인가요?"
answer = openai_rag_chain.invoke(question)

print(f"쿼리: {question}")
print(f"답변: {answer}") 

---

## **Langfuse를 활용한 평가**

1. **환경 설정**: Langfuse 클라이언트 초기화 및 인증
2. **데이터셋 업로드**: `create_dataset()` 및 `create_dataset_item()` 사용
3. **모니터링 RAG 체인**: `CallbackHandler`로 자동 추적 설정
4. **평가 실행**: 데이터셋 기반 체계적 평가

**주요 장점:**
- 🔄 자동화된 추적 및 로깅
- 📊 시각적 대시보드 제공
- 🔍 다양한 평가 지표 지원
- 🚀 확장 가능한 평가 파이프라인

---
### 1) **Langfuse 환경 설정** 

In [ ]:
from langfuse.langchain import CallbackHandler

# Langfuse 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

In [ ]:
from langfuse import get_client

# Langfuse 클라이언트 초기화
langfuse_client = get_client()

# 인증 확인
print("Langfuse 인증 상태:", langfuse_client.auth_check())

---
### 2) **평가용 데이터셋 업로드** 

In [ ]:
df_qa_test.head(2)

In [ ]:
# Langfuse에서 데이터셋 생성
name = "RAG_Evaluation_Dataset_Test"
dataset = langfuse_client.create_dataset(name="RAG_Evaluation_Dataset_Test")

print(f"생성된 데이터셋: {dataset.name}")

In [ ]:
# 평가용 데이터셋을 변환 
data=[
    {
        "user_input": row["user_input"],
        "reference": row["reference"],
        "reference_contexts": row["reference_contexts"],
    } for _, row in df_qa_test.iterrows()
]

print(f"평가용 데이터셋 아이템 수: {len(data)}개")

In [ ]:
# 데이터셋 아이템 추가
for item in data:
    langfuse_client.create_dataset_item(
        dataset_name=name,
        input=item.get("user_input", ""),
        expected_output=item.get("reference", ""),
        metadata={
            "reference_contexts": item.get("reference_contexts", ""),            }
    )


# langfuse에 플러시 (저장)
langfuse_client.flush()

In [ ]:
# 추가된 데이터셋 확인 (가져오기)
dataset = langfuse_client.get_dataset(name="RAG_Evaluation_Dataset_Test")
print(f"생성된 데이터셋: {dataset.name}")
print(f"데이터셋 아이템 수: {len(dataset.items)}개")

In [ ]:
# 평가용 데이터셋 아이템 출력
for item in dataset.items[:5]:  # 처음 5개 아이템만 출력
    print(f"입력: {item.input}")
    print(f"기대 출력: {item.expected_output}")
    print(f"메타데이터: {item.metadata}")
    print("-"*200)

---
### 3) **데이터셋 기반 평가 실행** 

In [ ]:
from langchain.evaluation import load_evaluator
from korouge_score import rouge_scorer
from krag.tokenizers import KiwiTokenizer

# Kiwi 토크나이저 사용하여 토큰화하는 클래스 정의 
class CustomKiwiTokenizer(KiwiTokenizer):
    def tokenize(self, text):
        return [t.form for t in super().tokenize(text)]

# 토크나이저 생성
kiwi_tokenizer = CustomKiwiTokenizer(model_type='knlm', typos='basic')

# ROUGE 스코어 계산
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], 
    tokenizer=kiwi_tokenizer      # tokenize 메소드를 갖는 토크나이저 사용
)

# 평가자 로드 (간결성 평가)
conciseness_evaluator = load_evaluator(
    evaluator="labeled_criteria", 
    criteria="conciseness",
    llm=llm
)

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from langfuse.langchain import CallbackHandler
from langfuse import get_client

@dataclass
class EvaluationResult:
    """평가 결과를 담는 데이터 클래스"""
    item_id: str
    input: Any
    output: str
    expected_output: str
    scores: Dict[str, float]
    details: Dict[str, Any]
    trace_id: Optional[str] = None
    error: Optional[str] = None

def run_dataset_evaluation(dataset_name: str, rag_chain, run_name: str) -> List[EvaluationResult]:
    """데이터셋 전체에 대한 평가 실행"""

    # 데이터셋 가져오기
    langfuse_client = get_client()
    dataset = langfuse_client.get_dataset(name=dataset_name)
    if not dataset:
        raise ValueError(f"데이터셋 '{dataset_name}'이(가) 존재하지 않습니다.")
    
    print(f"📊 RAG 평가 시작: {dataset_name} ({len(dataset.items)}개 항목)")
    
    results = []
    successful = 0
    failed = 0
    
    for idx, item in enumerate(dataset.items, 1):
        try:
            print(f"\n🔄 아이템 {idx}/{len(dataset.items)} 처리 중...")
            
            # Langfuse 트레이싱 설정
            with item.run(run_name=run_name) as root_span:

                # RAG 체인 실행
                output = rag_chain.invoke(
                    item.input,
                    config={"callbacks": [CallbackHandler()]}  # Langfuse 콜백 핸들러 추가
                    )
                
                # 평가 수행
                scores, details = {}, {}

                # 1. ROUGE 점수 평가
                try:
                    rouge_results = scorer.score(
                        str(item.expected_output), 
                        str(output)
                    )
                    rouge_scores = {
                        "rouge1": rouge_results['rouge1'].fmeasure,
                        "rouge2": rouge_results['rouge2'].fmeasure,
                        "rougeL": rouge_results['rougeL'].fmeasure
                    }
                    scores["rouge"] = sum(rouge_scores.values()) / len(rouge_scores)
                    details["rouge"] = rouge_scores
                except Exception as e:
                    scores["rouge"] = 0.0
                    details["rouge"] = {"error": str(e)}

                # 2. 간결성 평가
                try:
                    conciseness_result = conciseness_evaluator.evaluate_strings(
                        input=str(item.input),
                        prediction=str(output),
                        reference=str(item.expected_output)
                    )
                    scores["conciseness"] = float(conciseness_result.get('score', 0))
                    details["conciseness"] = {
                        "reasoning": conciseness_result.get('reasoning', ''),
                        "score": conciseness_result.get('score', 0)
                    }
                except Exception as e:
                    scores["conciseness"] = 0.0
                    details["conciseness"] = {"error": str(e)}

                # 전체 점수 계산 및 기록
                overall_score = sum(scores.values()) / len(scores)
                root_span.score(name="overall", value=overall_score)
                
                # 각 평가 점수 기록
                for score_name, score_value in scores.items():
                    root_span.score(name=score_name, value=score_value)
                
                # 결과 저장
                result = EvaluationResult(
                    item_id=item.id,
                    input=item.input,
                    output=str(output),
                    expected_output=str(item.expected_output) if item.expected_output else "",
                    scores=scores,
                    details=details,
                    trace_id=getattr(root_span, 'trace_id', None)
                )
                results.append(result)
                successful += 1
                
                print(f"   ✅ 완료 (종합 점수: {overall_score:.2f})")
                print(f"   🔍 세부 정보: {details}")
                
        except Exception as e:
            failed += 1
            print(f"   ❌ 실패: {str(e)}")
            
            # 실패해도 결과에 기록
            results.append(EvaluationResult(
                item_id=item.id,
                input=item.input,
                output="",
                expected_output=str(item.expected_output) if item.expected_output else "",
                scores={},
                details={},
                error=str(e)
            ))
    
    # 결과 요약
    print(f"\n📋 평가 완료: 성공 {successful}개, 실패 {failed}개")
    
    return results


In [ ]:
# 평가 실행
results = run_dataset_evaluation(
    dataset_name="RAG_Evaluation_Dataset_Test",  # 평가할 데이터셋 이름
    rag_chain=openai_rag_chain,  # 실제 RAG 체인
    run_name="simple_evaluation_v1" # 평가 실행 이름
)

---
### **[실습]**  

- Langfuse에 설정한 테스트 데이터에 대한 RAG를 수행할 수 있는 체인을 정의합니다. 
- LangChain의 `load_evaluator`를 활용하여 평가자를 로드하고, LLM-as-Judge 방식으로 생성된 답변의 품질을 평가합니다. 
- 각 답변의 평가 결과를 Langfuse에 기록합니다.
- Langfuse 대시보드에서 평가 결과를 시각적으로 분석합니다. 

In [ ]:
# rag_bot 함수 정의
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict

def rag_bot(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""문서 기반 질의응답 어시스턴트입니다.
- 제공된 문서만 참고하여 답변
- 불확실할 경우 '모르겠습니다' 라고 응답
- 3문장 이내로 답변

[문서]
{context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[질문]{question}\n\n[답변]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

# 모델별 비교
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

gpt_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Reference-free 평가
def reference_free_evaluation():
    # 사용자 정의 평가 기준
    custom_criteria = {
        "conciseness": "불필요한 반복이나 장황함 없이 핵심 내용을 전달하는가?",
        "helpfulness": "실질적인 도움이 되는 정도는 어떠한가?",
        "harmfulness": "해로운 내용이 포함되어 있지 않은가?"
    }
    
    evaluator = load_evaluator(
        "pairwise_string",
        criteria=custom_criteria,
        llm=ChatOpenAI(model="gpt-4.1", temperature=0),
        callbacks=[langfuse_handler]
    )
    
    for idx, row in df_qa_test.iterrows():
        question = row['user_input']
        
        gpt_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gpt_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        gemini_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gemini_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        result = evaluator.evaluate_string_pairs(
            prediction=gpt_response["answer"],
            prediction_b=gemini_response["answer"],
            input=question
        )
        
        print(f"Q{idx+1}: {result['value']} 승리")

# Reference-based 평가
def reference_based_evaluation():
    # 사용자 정의 평가 기준
    custom_criteria = {
        "conciseness": "불필요한 반복이나 장황함 없이 핵심 내용을 전달하는가?",
        "helpfulness": "실질적인 도움이 되는 정도는 어떠한가?",
        "harmfulness": "해로운 내용이 포함되어 있지 않은가?"
    }
    
    evaluator = load_evaluator(
        "labeled_pairwise_string",
        criteria=custom_criteria,
        llm=ChatOpenAI(model="gpt-4.1", temperature=0),
        callbacks=[langfuse_handler]
    )
    
    for idx, row in df_qa_test.iterrows():
        question = row['user_input']
        reference = row['reference']
        
        gpt_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gpt_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        gemini_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gemini_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        result = evaluator.evaluate_string_pairs(
            prediction=gpt_response["answer"],
            prediction_b=gemini_response["answer"],
            input=question,
            reference=reference
        )
        
        print(f"Q{idx+1}: {result['value']} 승리")

print("Reference-free 평가 시작")
reference_free_evaluation()

print("Reference-based 평가 시작")  
reference_based_evaluation()

In [36]:
# 추가 평가자 로드
from langchain.evaluation import load_evaluator

relevance_evaluator = load_evaluator(
    evaluator="labeled_criteria", 
    criteria="relevance",
    llm=llm
)

correctness_evaluator = load_evaluator(
    evaluator="labeled_criteria", 
    criteria="correctness",
    llm=llm
)

helpfulness_evaluator = load_evaluator(
    evaluator="labeled_criteria", 
    criteria="helpfulness",
    llm=llm
)

print("평가자 로드 완료: 관련성, 정확성, 도움성, 간결성")


def run_comprehensive_evaluation(dataset_name: str, rag_chain, run_name: str) -> List[EvaluationResult]:
    """포괄적인 RAG 평가 실행"""
    
    langfuse_client = get_client()
    dataset = langfuse_client.get_dataset(name=dataset_name)
    if not dataset:
        raise ValueError(f"데이터셋 '{dataset_name}'이(가) 존재하지 않습니다.")
    
    print(f"RAG 평가 시작: {dataset_name} ({len(dataset.items)}개 항목)")
    
    results = []
    successful = 0
    failed = 0
    
    for idx, item in enumerate(dataset.items, 1):
        try:
            print(f"아이템 {idx}/{len(dataset.items)} 처리 중...")
            
            with item.run(run_name=run_name) as root_span:
                
                output = rag_chain.invoke(
                    item.input,
                    config={"callbacks": [CallbackHandler()]}
                )
                
                scores, details = {}, {}
                
                # ROUGE 점수 평가
                try:
                    rouge_results = scorer.score(
                        str(item.expected_output), 
                        str(output)
                    )
                    rouge_scores = {
                        "rouge1": rouge_results['rouge1'].fmeasure,
                        "rouge2": rouge_results['rouge2'].fmeasure,
                        "rougeL": rouge_results['rougeL'].fmeasure
                    }
                    scores["rouge"] = sum(rouge_scores.values()) / len(rouge_scores)
                    details["rouge"] = rouge_scores
                except Exception as e:
                    scores["rouge"] = 0.0
                    details["rouge"] = {"error": str(e)}
                
                # 관련성 평가
                try:
                    relevance_result = relevance_evaluator.evaluate_strings(
                        input=str(item.input),
                        prediction=str(output),
                        reference=str(item.expected_output)
                    )
                    scores["relevance"] = float(relevance_result.get('score', 0))
                    details["relevance"] = {
                        "reasoning": relevance_result.get('reasoning', ''),
                        "score": relevance_result.get('score', 0)
                    }
                except Exception as e:
                    scores["relevance"] = 0.0
                    details["relevance"] = {"error": str(e)}
                
                # 정확성 평가
                try:
                    correctness_result = correctness_evaluator.evaluate_strings(
                        input=str(item.input),
                        prediction=str(output),
                        reference=str(item.expected_output)
                    )
                    scores["correctness"] = float(correctness_result.get('score', 0))
                    details["correctness"] = {
                        "reasoning": correctness_result.get('reasoning', ''),
                        "score": correctness_result.get('score', 0)
                    }
                except Exception as e:
                    scores["correctness"] = 0.0
                    details["correctness"] = {"error": str(e)}
                
                # 도움성 평가
                try:
                    helpfulness_result = helpfulness_evaluator.evaluate_strings(
                        input=str(item.input),
                        prediction=str(output),
                        reference=str(item.expected_output)
                    )
                    scores["helpfulness"] = float(helpfulness_result.get('score', 0))
                    details["helpfulness"] = {
                        "reasoning": helpfulness_result.get('reasoning', ''),
                        "score": helpfulness_result.get('score', 0)
                    }
                except Exception as e:
                    scores["helpfulness"] = 0.0
                    details["helpfulness"] = {"error": str(e)}
                
                # 간결성 평가
                try:
                    conciseness_result = conciseness_evaluator.evaluate_strings(
                        input=str(item.input),
                        prediction=str(output),
                        reference=str(item.expected_output)
                    )
                    scores["conciseness"] = float(conciseness_result.get('score', 0))
                    details["conciseness"] = {
                        "reasoning": conciseness_result.get('reasoning', ''),
                        "score": conciseness_result.get('score', 0)
                    }
                except Exception as e:
                    scores["conciseness"] = 0.0
                    details["conciseness"] = {"error": str(e)}
                
                overall_score = sum(scores.values()) / len(scores) if scores else 0.0
                root_span.score(name="overall", value=overall_score)
                
                for score_name, score_value in scores.items():
                    root_span.score(name=score_name, value=score_value)
                
                result = EvaluationResult(
                    item_id=item.id,
                    input=item.input,
                    output=str(output),
                    expected_output=str(item.expected_output) if item.expected_output else "",
                    scores=scores,
                    details=details,
                    trace_id=getattr(root_span, 'trace_id', None)
                )
                results.append(result)
                successful += 1
                
                print(f"완료 (종합 점수: {overall_score:.3f})")
                
        except Exception as e:
            failed += 1
            print(f"실패: {str(e)}")
            
            results.append(EvaluationResult(
                item_id=item.id,
                input=item.input,
                output="",
                expected_output=str(item.expected_output) if item.expected_output else "",
                scores={},
                details={},
                error=str(e)
            ))
    
    print(f"평가 완료: 성공 {successful}개, 실패 {failed}개")
    
    return results


# 포괄적 평가 실행
comprehensive_results = run_comprehensive_evaluation(
    dataset_name="RAG_Evaluation_Dataset_Test",
    rag_chain=openai_rag_chain,
    run_name="comprehensive_evaluation_v5"
)


def analyze_results(results: List[EvaluationResult]):
    """평가 결과 분석"""
    
    if not results:
        print("분석할 결과가 없습니다.")
        return
    
    successful_results = [r for r in results if not r.error and r.scores]
    
    if not successful_results:
        print("성공한 평가 결과가 없습니다.")
        return
    
    print(f"평가 결과 분석 ({len(successful_results)}개 성공)")
    print("=" * 60)
    
    metrics = ["rouge", "relevance", "correctness", "helpfulness", "conciseness"]
    metric_scores = {metric: [] for metric in metrics}
    
    for result in successful_results:
        for metric in metrics:
            if metric in result.scores:
                metric_scores[metric].append(result.scores[metric])
    
    print("평균 점수:")
    overall_scores = []
    for metric in metrics:
        if metric_scores[metric]:
            avg_score = np.mean(metric_scores[metric])
            std_score = np.std(metric_scores[metric])
            overall_scores.extend(metric_scores[metric])
            print(f"  {metric.upper():>12}: {avg_score:.3f} (±{std_score:.3f})")
    
    if overall_scores:
        total_avg = np.mean([np.mean([result.scores[m] for m in metrics if m in result.scores]) 
                           for result in successful_results])
        print(f"  {'OVERALL':>12}: {total_avg:.3f}")
    
    if successful_results:
        sorted_results = sorted(successful_results, 
                              key=lambda x: np.mean([x.scores[m] for m in metrics if m in x.scores]), 
                              reverse=True)
        
        print(f"최고 성능 사례:")
        best = sorted_results[0]
        best_avg = np.mean([best.scores[m] for m in metrics if m in best.scores])
        print(f"  질문: {best.input[:100]}...")
        print(f"  점수: {best_avg:.3f}")
        print(f"  답변: {best.output[:150]}...")
        
        print(f"개선 필요 사례:")
        worst = sorted_results[-1]
        worst_avg = np.mean([worst.scores[m] for m in metrics if m in worst.scores])
        print(f"  질문: {worst.input[:100]}...")
        print(f"  점수: {worst_avg:.3f}")
        print(f"  답변: {worst.output[:150]}...")

analyze_results(comprehensive_results)

평가자 로드 완료: 관련성, 정확성, 도움성, 간결성
Reference-free 평가 시작
Q1: A 승리
Q2: B 승리
Q3: A 승리
Q4: A 승리
Q5: B 승리
Q6: B 승리
Q7: A 승리
Q8: None 승리
Q9: A 승리
Q10: A 승리
Q11: A 승리
Q12: A 승리
Q13: B 승리
Q14: None 승리
Q15: B 승리
Q16: None 승리
Q17: A 승리
Q18: B 승리
Q19: A 승리
Q20: A 승리
Q21: A 승리
Q22: A 승리
Q23: B 승리
Q24: None 승리
Q25: A 승리
Q26: A 승리
Q27: B 승리
Q28: A 승리
Q29: B 승리
Q30: B 승리
Q31: A 승리
Q32: None 승리
Q33: B 승리
Q34: B 승리


KeyboardInterrupt: 